<a href="https://colab.research.google.com/github/hoangnguyen3101/Application-algorithms/blob/main/Lab_02_Chia_de_tri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab thực hành 2: Phương pháp chia để trị

**Môn học:** Thuật toán ứng dụng  
**Chủ đề:** Divide and Conquer — Chia để trị  

## Mục tiêu học tập

Sau khi hoàn thành bài lab này, sinh viên có thể:

- Giải thích được ba bước cơ bản của phương pháp chia để trị: **Divide – Conquer – Combine**.
- Viết và phân tích được phương trình truy hồi dạng:

$$
T(n) = aT\left(\frac{n}{b}\right) + f(n)
$$

- Cài đặt và giải thích được các thuật toán:
  - Tìm kiếm nhị phân;
  - Merge Sort;
  - Quick Sort;
  - Quick Sort ngẫu nhiên hóa.
- So sánh Merge Sort và Quick Sort trên các kiểu dữ liệu khác nhau.
- Nhận ra khi nào chia để trị hiệu quả và khi nào nên dùng cách tiếp cận khác.

---

## Cách sử dụng notebook

Mỗi phần gồm:

1. Nhắc lại ý tưởng lý thuyết.
2. Code mẫu hoàn chỉnh.
3. Ví dụ chạy thử.
4. Câu hỏi phân tích.
5. Bài tập TODO cho sinh viên.

Các ô có đánh dấu `TODO` là phần sinh viên cần hoàn thiện hoặc chỉnh sửa.

## 0. Chuẩn bị môi trường

Notebook chỉ dùng thư viện chuẩn của Python và `matplotlib` để vẽ minh họa.
Nếu chạy trên Google Colab hoặc môi trường chưa có `matplotlib`, hãy cài đặt trước khi chạy phần trực quan hóa.

In [ ]:
# Nếu chạy trên Google Colab hoặc môi trường chưa có matplotlib, bỏ dấu # ở dòng dưới.
!pip -q install matplotlib

In [ ]:
from dataclasses import dataclass
import math
import random
import time
from typing import List, Tuple, Optional, Any, Dict
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"
random.seed(42)

In [ ]:
def show_result(title, value):
    print("=" * 80)
    print(title)
    print("=" * 80)
    print(value)


@dataclass
class Counter:
    comparisons: int = 0
    swaps: int = 0
    calls: int = 0

    def reset(self):
        self.comparisons = 0
        self.swaps = 0
        self.calls = 0

## 1. Khuôn mẫu tư duy chia để trị

Một thuật toán chia để trị thường có dạng:

1. **Divide — Chia:** Chia bài toán kích thước lớn thành các bài toán con nhỏ hơn, cùng cấu trúc.
2. **Conquer — Trị:** Giải các bài toán con, thường bằng đệ quy.
3. **Combine — Kết hợp:** Ghép nghiệm của các bài toán con để tạo nghiệm của bài toán ban đầu.

Phương trình truy hồi tổng quát:

$$
T(n) = aT\left(\frac{n}{b}\right) + f(n),
$$

trong đó:

- $a$ là số bài toán con;
- $n/b$ là kích thước mỗi bài toán con;
- $f(n)$ là chi phí của bước chia và kết hợp.

Một số ví dụ kinh điển:

| Thuật toán | Divide | Conquer | Combine | Truy hồi |
|---|---|---|---|---|
| Binary Search | Chọn phần tử giữa | Tìm trong một nửa | Không cần | $T(n)=T(n/2)+\Theta(1)$ |
| Merge Sort | Chia đôi mảng | Sắp xếp hai nửa | Trộn hai nửa | $T(n)=2T(n/2)+\Theta(n)$ |
| Quick Sort | Phân hoạch theo pivot | Sắp xếp hai phần | Không cần | Trung bình $T(n)\approx 2T(n/2)+\Theta(n)$ |

In [ ]:
# Mã giả theo phong cách Python cho khuôn mẫu chia để trị.
# Các hàm is_base_case, solve_directly, divide, combine phụ thuộc vào từng bài toán cụ thể.

def divide_and_conquer_template(problem):
    if is_base_case(problem):
        return solve_directly(problem)

    subproblems = divide(problem)
    subsolutions = []

    for subproblem in subproblems:
        subsolutions.append(divide_and_conquer_template(subproblem))

    return combine(subsolutions)


show_result(
    "Khuôn mẫu chia để trị",
    "Base case -> Divide -> Recursive conquer -> Combine"
)

> Ô code trên chỉ là **mã giả theo phong cách Python**.  
> Không chạy trực tiếp nếu chưa định nghĩa các hàm `is_base_case`, `solve_directly`, `divide`, `combine`.

## 2. Phân tích hệ thức truy hồi

Trong bài giảng, ta quan tâm ba công cụ chính:

1. Phương pháp thế.
2. Cây đệ quy.
3. Định lý thợ.

Trong lab này, ta minh họa nhanh bằng cây đệ quy cho Merge Sort và một hàm hỗ trợ áp dụng trực giác của định lý thợ cho một số dạng phổ biến.

In [ ]:
def recursion_tree_merge_sort(n: int):
    # Minh họa chi phí từng tầng của Merge Sort khi n là lũy thừa của 2.
    # T(n) = 2T(n/2) + n.
    level = 0
    rows = []

    while True:
        num_nodes = 2 ** level
        subproblem_size = n // num_nodes
        cost_per_node = subproblem_size
        total_cost = num_nodes * cost_per_node
        rows.append((level, num_nodes, subproblem_size, total_cost))

        if subproblem_size == 1:
            break

        level += 1

    return rows


rows = recursion_tree_merge_sort(16)

print("level | số nút | kích thước mỗi bài toán con | tổng chi phí tầng")
for row in rows:
    print(f"{row[0]:>5} | {row[1]:>6} | {row[2]:>27} | {row[3]:>17}")

In [ ]:
def plot_merge_sort_recursion_tree_cost(n: int):
    rows = recursion_tree_merge_sort(n)
    levels = [r[0] for r in rows]
    costs = [r[3] for r in rows]

    plt.figure(figsize=(7, 4))
    plt.bar(levels, costs)
    plt.xlabel("Tầng trong cây đệ quy")
    plt.ylabel("Tổng chi phí ở tầng")
    plt.title(f"Chi phí từng tầng của Merge Sort với n = {n}")
    plt.xticks(levels)
    plt.show()


plot_merge_sort_recursion_tree_cost(32)

### Nhận xét

Với Merge Sort:

$$
T(n)=2T(n/2)+\Theta(n).
$$

Ở mỗi tầng của cây đệ quy, tổng chi phí trộn là $\Theta(n)$.  
Số tầng là $\log_2 n$.  

Do đó:

$$
T(n)=\Theta(n\log n).
$$

In [ ]:
def master_theorem_hint(a: int, b: int, f_description: str):
    # Hàm mô tả nhanh cách áp dụng định lý thợ cho một số ví dụ quen thuộc.
    # Đây không phải bộ giải biểu thức tổng quát.
    critical_exp = math.log(a, b)
    return {
        "a": a,
        "b": b,
        "critical_function": f"n^{critical_exp:.4f}",
        "f(n)": f_description,
        "note": "So sánh f(n) với n^(log_b a) để xác định trường hợp của định lý thợ."
    }


examples = [
    master_theorem_hint(1, 2, "Theta(1)"),
    master_theorem_hint(2, 2, "Theta(n)"),
    master_theorem_hint(4, 2, "Theta(n)"),
    master_theorem_hint(4, 2, "Theta(n^3)"),
]

for ex in examples:
    print(ex)

## 3. Tìm kiếm nhị phân

### Bài toán

Cho mảng $A$ đã sắp xếp tăng dần và khóa $k$.  
Tìm chỉ số $i$ sao cho $A[i]=k$. Nếu không tồn tại, trả về `-1`.

### Khuôn khổ chia để trị

- **Divide:** Chọn phần tử giữa.
- **Conquer:** Chỉ tiếp tục tìm ở nửa trái hoặc nửa phải.
- **Combine:** Không cần, vì nghiệm trả trực tiếp từ bài toán con.

Truy hồi:

$$
T(n)=T(n/2)+\Theta(1)=\Theta(\log n).
$$

In [ ]:
def binary_search_recursive(A: List[int], key: int, left: int = 0, right: Optional[int] = None, trace: bool = False) -> int:
    if right is None:
        right = len(A) - 1

    if trace:
        print(f"Tìm trong đoạn [{left}, {right}]")

    if left > right:
        return -1

    mid = (left + right) // 2

    if trace:
        print(f"  mid = {mid}, A[mid] = {A[mid]}")

    if A[mid] == key:
        return mid
    elif key < A[mid]:
        return binary_search_recursive(A, key, left, mid - 1, trace)
    else:
        return binary_search_recursive(A, key, mid + 1, right, trace)


A = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
key = 23

idx = binary_search_recursive(A, key, trace=True)
show_result(f"Tìm kiếm nhị phân khóa {key}", f"Vị trí tìm thấy: {idx}")

In [ ]:
def binary_search_iterative(A: List[int], key: int, trace: bool = False) -> int:
    left, right = 0, len(A) - 1

    while left <= right:
        mid = (left + right) // 2

        if trace:
            print(f"[{left}, {right}], mid = {mid}, A[mid] = {A[mid]}")

        if A[mid] == key:
            return mid
        elif key < A[mid]:
            right = mid - 1
        else:
            left = mid + 1

    return -1


show_result(
    "Phiên bản lặp",
    binary_search_iterative(A, 56, trace=True)
)

### TODO 1

Hoàn thiện hàm `binary_search_count` để trả về:

- vị trí tìm thấy;
- số lần so sánh với phần tử giữa.

Gợi ý: mỗi lần xét `A[mid]` thì tăng biến đếm.

In [ ]:
def binary_search_count(A: List[int], key: int) -> Tuple[int, int]:
    # TODO: Có thể yêu cầu sinh viên tự hoàn thiện lại phần này.
    left, right = 0, len(A) - 1
    comparisons = 0

    while left <= right:
        mid = (left + right) // 2
        comparisons += 1

        if A[mid] == key:
            return mid, comparisons
        elif key < A[mid]:
            right = mid - 1
        else:
            left = mid + 1

    return -1, comparisons


for key in [2, 23, 91, 100]:
    idx, comps = binary_search_count(A, key)
    print(f"key = {key:>3}, index = {idx:>2}, comparisons = {comps}")

## 4. Merge Sort

### Ý tưởng

Merge Sort sắp xếp mảng bằng cách:

1. Chia mảng thành hai nửa.
2. Sắp xếp đệ quy từng nửa.
3. Trộn hai nửa đã sắp xếp.

### Truy hồi

$$
T(n)=2T(n/2)+\Theta(n)=\Theta(n\log n).
$$

Merge Sort có độ phức tạp $\Theta(n\log n)$ trong mọi trường hợp, nhưng cần thêm bộ nhớ phụ $O(n)$.

In [ ]:
def merge(left: List[int], right: List[int], counter: Optional[Counter] = None, trace: bool = False) -> List[int]:
    i = j = 0
    result = []

    if trace:
        print(f"MERGE {left} và {right}")

    while i < len(left) and j < len(right):
        if counter:
            counter.comparisons += 1

        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

        if trace:
            print("  ", result)

    result.extend(left[i:])
    result.extend(right[j:])

    if trace:
        print("Kết quả:", result)

    return result


merge([2, 4, 5], [1, 3, 6], trace=True)

In [ ]:
def merge_sort(A: List[int], counter: Optional[Counter] = None, trace: bool = False, depth: int = 0) -> List[int]:
    if counter:
        counter.calls += 1

    if trace:
        print("  " * depth + f"merge_sort({A})")

    if len(A) <= 1:
        return A[:]

    mid = len(A) // 2
    left_sorted = merge_sort(A[:mid], counter, trace, depth + 1)
    right_sorted = merge_sort(A[mid:], counter, trace, depth + 1)

    return merge(left_sorted, right_sorted, counter, trace)


A_demo = [5, 2, 4, 6, 1, 3, 2, 6]
counter = Counter()
sorted_A = merge_sort(A_demo, counter=counter, trace=True)

show_result(
    "Merge Sort",
    {
        "input": A_demo,
        "output": sorted_A,
        "comparisons": counter.comparisons,
        "recursive_calls": counter.calls
    }
)

In [ ]:
def visualize_merge_sort_splitting(A: List[int], depth: int = 0, rows: Optional[List[Tuple[int, List[int]]]] = None):
    # Lưu lại các mảng con xuất hiện trong quá trình chia của Merge Sort.
    if rows is None:
        rows = []

    rows.append((depth, A[:]))

    if len(A) <= 1:
        return rows

    mid = len(A) // 2
    visualize_merge_sort_splitting(A[:mid], depth + 1, rows)
    visualize_merge_sort_splitting(A[mid:], depth + 1, rows)

    return rows


rows = visualize_merge_sort_splitting(A_demo)

for depth, arr in rows:
    print("  " * depth + str(arr))

### TODO 2

Với mảng:

```python
A = [3, 1, 4, 1, 5, 9, 2, 6]
```

Hãy:

1. Chạy Merge Sort.
2. In cây chia đệ quy.
3. Đếm số phép so sánh trong bước merge.
4. Nhận xét vì sao số tầng của cây đệ quy xấp xỉ $\log_2 n$.

In [ ]:
# TODO 2: Sinh viên chạy và phân tích trên mảng sau
A_todo = [3, 1, 4, 1, 5, 9, 2, 6]

counter = Counter()
print("Cây chia:")
for depth, arr in visualize_merge_sort_splitting(A_todo):
    print("  " * depth + str(arr))

print("\nKết quả sắp xếp:")
print(merge_sort(A_todo, counter=counter))

print("\nSố phép so sánh:", counter.comparisons)

## 5. Quick Sort

### Ý tưởng

Quick Sort sắp xếp mảng tại chỗ bằng cách:

1. Chọn một phần tử làm chốt `pivot`.
2. Phân hoạch mảng:
   - bên trái chốt: các phần tử không lớn hơn chốt;
   - bên phải chốt: các phần tử lớn hơn chốt.
3. Sắp xếp đệ quy hai phần còn lại.

### Điểm khác với Merge Sort

- Merge Sort chia trước, trộn sau.
- Quick Sort phân hoạch trước, sau đó không cần bước combine.

### Độ phức tạp

- Tốt nhất và trung bình: $O(n\log n)$.
- Xấu nhất: $\Theta(n^2)$ nếu phân hoạch quá lệch.

In [ ]:
def partition(A: List[int], low: int, high: int, counter: Optional[Counter] = None, trace: bool = False) -> int:
    pivot = A[high]
    i = low - 1

    if trace:
        print(f"PARTITION đoạn A[{low}:{high + 1}] = {A[low:high + 1]}, pivot = {pivot}")

    for j in range(low, high):
        if counter:
            counter.comparisons += 1

        if trace:
            print(f"  j = {j}, A[j] = {A[j]}, so sánh với pivot {pivot}")

        if A[j] <= pivot:
            i += 1
            A[i], A[j] = A[j], A[i]
            if counter:
                counter.swaps += 1
            if trace:
                print(f"    swap A[{i}] và A[{j}] -> {A}")

    A[i + 1], A[high] = A[high], A[i + 1]
    if counter:
        counter.swaps += 1

    if trace:
        print(f"Đưa pivot về vị trí {i + 1} -> {A}")

    return i + 1


A_partition = [2, 8, 7, 1, 3, 5, 6, 4]
counter = Counter()
q = partition(A_partition, 0, len(A_partition) - 1, counter=counter, trace=True)

show_result(
    "Kết quả PARTITION",
    {
        "array": A_partition,
        "pivot_index": q,
        "comparisons": counter.comparisons,
        "swaps": counter.swaps
    }
)

In [ ]:
def quick_sort(A: List[int], low: int = 0, high: Optional[int] = None, counter: Optional[Counter] = None, trace: bool = False):
    if high is None:
        high = len(A) - 1

    if counter:
        counter.calls += 1

    if low < high:
        q = partition(A, low, high, counter, trace)

        if trace:
            print(f"Quick Sort trái: A[{low}:{q}]")
        quick_sort(A, low, q - 1, counter, trace)

        if trace:
            print(f"Quick Sort phải: A[{q + 1}:{high + 1}]")
        quick_sort(A, q + 1, high, counter, trace)


A_demo = [13, 19, 9, 5, 12, 8, 7, 4, 21, 2, 6, 11]
counter = Counter()
quick_sort(A_demo, counter=counter, trace=True)

show_result(
    "Quick Sort",
    {
        "sorted": A_demo,
        "comparisons": counter.comparisons,
        "swaps": counter.swaps,
        "recursive_calls": counter.calls
    }
)

### TODO 3

Chạy `PARTITION` trên mảng sau, với chốt là phần tử cuối:

```python
A = [13, 19, 9, 5, 12, 8, 7, 4, 21, 2, 6, 11]
```

Yêu cầu:

1. Cho biết vị trí cuối cùng của chốt.
2. In mảng sau khi phân hoạch.
3. Giải thích vì sao các phần tử bên trái chốt đều không lớn hơn chốt.

In [ ]:
# TODO 3
A_todo = [13, 19, 9, 5, 12, 8, 7, 4, 21, 2, 6, 11]
counter = Counter()
q = partition(A_todo, 0, len(A_todo) - 1, counter=counter, trace=True)

print("Mảng sau partition:", A_todo)
print("Vị trí pivot:", q)
print("Pivot:", A_todo[q])

## 6. Quick Sort ngẫu nhiên hóa

Nếu luôn chọn phần tử cuối làm pivot, dữ liệu đã sắp xếp có thể làm Quick Sort rơi vào trường hợp xấu nhất.

Ý tưởng cải tiến:

1. Chọn ngẫu nhiên một vị trí pivot trong đoạn đang xét.
2. Đổi pivot đó với phần tử cuối.
3. Gọi lại thủ tục `partition`.

Khi đó, với mọi đầu vào cố định, thời gian kỳ vọng là $O(n\log n)$.

In [ ]:
def randomized_partition(A: List[int], low: int, high: int, counter: Optional[Counter] = None, trace: bool = False) -> int:
    pivot_index = random.randint(low, high)
    A[pivot_index], A[high] = A[high], A[pivot_index]

    if counter:
        counter.swaps += 1

    if trace:
        print(f"Chọn pivot ngẫu nhiên tại vị trí {pivot_index}, đưa về cuối.")

    return partition(A, low, high, counter, trace)


def randomized_quick_sort(A: List[int], low: int = 0, high: Optional[int] = None, counter: Optional[Counter] = None):
    if high is None:
        high = len(A) - 1

    if counter:
        counter.calls += 1

    if low < high:
        q = randomized_partition(A, low, high, counter)
        randomized_quick_sort(A, low, q - 1, counter)
        randomized_quick_sort(A, q + 1, high, counter)


A_sorted = list(range(1, 21))

A1 = A_sorted[:]
c1 = Counter()
quick_sort(A1, counter=c1)

A2 = A_sorted[:]
c2 = Counter()
randomized_quick_sort(A2, counter=c2)

show_result(
    "So sánh Quick Sort cố định pivot và ngẫu nhiên pivot trên mảng đã sắp xếp",
    {
        "fixed_pivot_comparisons": c1.comparisons,
        "random_pivot_comparisons": c2.comparisons,
        "fixed_pivot_sorted": A1,
        "random_pivot_sorted": A2
    }
)

## 7. Thí nghiệm so sánh Merge Sort và Quick Sort

Ta sẽ so sánh số phép so sánh của Merge Sort, Quick Sort cố định pivot và Quick Sort ngẫu nhiên pivot trên ba kiểu dữ liệu:

1. Dữ liệu ngẫu nhiên.
2. Dữ liệu đã sắp xếp tăng.
3. Dữ liệu sắp xếp giảm.

Lưu ý: Đây là thí nghiệm nhỏ trong Python để quan sát xu hướng, không phải benchmark tuyệt đối.

In [ ]:
def run_sort_experiment(n: int, data_type: str) -> Dict[str, Any]:
    if data_type == "random":
        data = list(range(n))
        random.shuffle(data)
    elif data_type == "sorted":
        data = list(range(n))
    elif data_type == "reversed":
        data = list(range(n, 0, -1))
    else:
        raise ValueError("data_type phải là random, sorted hoặc reversed")

    A_merge = data[:]
    c_merge = Counter()
    merge_sort(A_merge, counter=c_merge)

    A_quick = data[:]
    c_quick = Counter()
    quick_sort(A_quick, counter=c_quick)

    A_rand = data[:]
    c_rand = Counter()
    randomized_quick_sort(A_rand, counter=c_rand)

    return {
        "n": n,
        "data_type": data_type,
        "merge_comparisons": c_merge.comparisons,
        "quick_comparisons": c_quick.comparisons,
        "random_quick_comparisons": c_rand.comparisons
    }


results = []
for data_type in ["random", "sorted", "reversed"]:
    for n in [20, 40, 80, 160]:
        results.append(run_sort_experiment(n, data_type))

for row in results:
    print(row)

In [ ]:
def plot_comparison(results, data_type: str):
    rows = [r for r in results if r["data_type"] == data_type]
    ns = [r["n"] for r in rows]

    merge_values = [r["merge_comparisons"] for r in rows]
    quick_values = [r["quick_comparisons"] for r in rows]
    random_quick_values = [r["random_quick_comparisons"] for r in rows]

    plt.figure(figsize=(7, 4))
    plt.plot(ns, merge_values, marker="o", label="Merge Sort")
    plt.plot(ns, quick_values, marker="o", label="Quick Sort pivot cuối")
    plt.plot(ns, random_quick_values, marker="o", label="Randomized Quick Sort")
    plt.xlabel("Kích thước n")
    plt.ylabel("Số phép so sánh")
    plt.title(f"So sánh trên dữ liệu: {data_type}")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_comparison(results, "random")
plot_comparison(results, "sorted")
plot_comparison(results, "reversed")

### Câu hỏi thảo luận

1. Trên dữ liệu ngẫu nhiên, Quick Sort và Merge Sort khác nhau như thế nào?
2. Trên dữ liệu đã sắp xếp, vì sao Quick Sort chọn pivot cuối bị xấu?
3. Quick Sort ngẫu nhiên hóa có làm kết quả sắp xếp thay đổi không?
4. Nếu cần thuật toán sắp xếp ổn định, nên chọn Merge Sort hay Quick Sort?

## 8. Ứng dụng: Bài toán kiểm tra hai số có tổng bằng x

### Bài toán

Cho tập $S$ gồm $n$ số nguyên và số nguyên $x$.  
Kiểm tra xem có tồn tại hai phần tử $a,b \in S$ sao cho:

$$
a+b=x.
$$

### Cách 1: Duyệt vét cạn

Kiểm tra mọi cặp phần tử.  
Độ phức tạp:

$$
O(n^2).
$$

### Cách 2: Sắp xếp rồi tìm kiếm

Ta có thể:

1. Sắp xếp mảng bằng Merge Sort hoặc Quick Sort.
2. Với mỗi phần tử $a$, tìm $x-a$ bằng Binary Search.

Độ phức tạp:

$$
O(n\log n).
$$

Một cách khác là sau khi sắp xếp, dùng hai con trỏ trái/phải, cũng đạt $O(n\log n)$ do chi phí sắp xếp.

In [ ]:
def two_sum_bruteforce(S: List[int], x: int) -> Optional[Tuple[int, int]]:
    n = len(S)

    for i in range(n):
        for j in range(i + 1, n):
            if S[i] + S[j] == x:
                return S[i], S[j]

    return None


def two_sum_sort_binary_search(S: List[int], x: int) -> Optional[Tuple[int, int]]:
    A = merge_sort(S)

    for i, a in enumerate(A):
        b = x - a

        # Tìm b trong phần bên phải i để tránh dùng lại cùng một phần tử.
        idx = binary_search_iterative(A[i + 1:], b)

        if idx != -1:
            return a, b

    return None


def two_sum_two_pointers(S: List[int], x: int) -> Optional[Tuple[int, int]]:
    A = merge_sort(S)
    left, right = 0, len(A) - 1

    while left < right:
        current = A[left] + A[right]

        if current == x:
            return A[left], A[right]
        elif current < x:
            left += 1
        else:
            right -= 1

    return None


S = [10, 4, 7, 1, 3, 8, 12]
x = 11

show_result(
    "Bài toán two-sum",
    {
        "bruteforce": two_sum_bruteforce(S, x),
        "sort_binary_search": two_sum_sort_binary_search(S, x),
        "two_pointers": two_sum_two_pointers(S, x)
    }
)

### TODO 4

Sinh viên tự tạo một danh sách gồm ít nhất 20 số nguyên.  
Sau đó:

1. Chạy ba cách giải two-sum ở trên.
2. So sánh thời gian chạy khi $n$ tăng.
3. Giải thích vì sao cách sắp xếp trước rồi tìm kiếm tốt hơn vét cạn khi $n$ lớn.

In [ ]:
# TODO 4: Thí nghiệm thời gian chạy two-sum

def measure_time(func, S, x, repeat=3):
    total = 0.0

    for _ in range(repeat):
        start = time.perf_counter()
        func(S, x)
        total += time.perf_counter() - start

    return total / repeat


sizes = [100, 300, 500, 1000]
time_rows = []

for n in sizes:
    S = list(range(n))
    random.shuffle(S)
    x = 2 * n + 1  # thường không có nghiệm, buộc thuật toán kiểm tra nhiều hơn

    time_rows.append({
        "n": n,
        "bruteforce": measure_time(two_sum_bruteforce, S, x),
        "sort_binary_search": measure_time(two_sum_sort_binary_search, S, x),
        "two_pointers": measure_time(two_sum_two_pointers, S, x)
    })

for row in time_rows:
    print(row)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot([r["n"] for r in time_rows], [r["bruteforce"] for r in time_rows], marker="o", label="Bruteforce O(n^2)")
plt.plot([r["n"] for r in time_rows], [r["sort_binary_search"] for r in time_rows], marker="o", label="Sort + Binary Search")
plt.plot([r["n"] for r in time_rows], [r["two_pointers"] for r in time_rows], marker="o", label="Sort + Two Pointers")
plt.xlabel("Kích thước n")
plt.ylabel("Thời gian trung bình")
plt.title("So sánh thời gian chạy bài toán two-sum")
plt.legend()
plt.grid(True)
plt.show()

## 9. Mở rộng nâng cao: Thuật toán Karatsuba

Nhân hai số nguyên lớn theo cách thông thường có độ phức tạp xấp xỉ $O(n^2)$ với $n$ là số chữ số.

Karatsuba dùng ý tưởng chia để trị:

Giả sử:

$$
x = a \cdot 10^m + b,
$$

$$
y = c \cdot 10^m + d.
$$

Cách nhân thông thường cần tính bốn tích:

$$
ac,\ ad,\ bc,\ bd.
$$

Karatsuba giảm xuống còn ba tích:

$$
z_0 = bd,
$$

$$
z_2 = ac,
$$

$$
z_1 = (a+b)(c+d)-z_2-z_0.
$$

Sau đó:

$$
xy = z_2 \cdot 10^{2m} + z_1 \cdot 10^m + z_0.
$$

Truy hồi:

$$
T(n)=3T(n/2)+\Theta(n)=\Theta(n^{\log_2 3}) \approx \Theta(n^{1.585}).
$$

In [ ]:
def karatsuba(x: int, y: int) -> int:
    # Cài đặt Karatsuba đơn giản cho số nguyên không âm.
    if x < 10 or y < 10:
        return x * y

    n = max(len(str(x)), len(str(y)))
    m = n // 2

    power = 10 ** m

    a, b = divmod(x, power)
    c, d = divmod(y, power)

    z0 = karatsuba(b, d)
    z2 = karatsuba(a, c)
    z1 = karatsuba(a + b, c + d) - z2 - z0

    return z2 * (10 ** (2 * m)) + z1 * power + z0


x = 12345678
y = 87654321

show_result(
    "Karatsuba",
    {
        "x": x,
        "y": y,
        "karatsuba": karatsuba(x, y),
        "python_builtin": x * y,
        "correct": karatsuba(x, y) == x * y
    }
)

### TODO 5

1. Tính độ phức tạp của Karatsuba bằng định lý thợ.
2. So sánh $\Theta(n^{\log_2 3})$ với $O(n^2)$.
3. Giải thích vì sao giảm từ 4 phép nhân con xuống 3 phép nhân con lại quan trọng.

## 10. Tổng kết bài lab

Qua bài lab này, sinh viên đã thực hành các nội dung chính:

- Chia để trị gồm ba bước: **Divide – Conquer – Combine**.
- Phân tích thuật toán chia để trị thường dẫn đến hệ thức truy hồi.
- Binary Search có độ phức tạp $\Theta(\log n)$.
- Merge Sort có độ phức tạp $\Theta(n\log n)$ trong mọi trường hợp, nhưng cần thêm bộ nhớ phụ.
- Quick Sort thường nhanh trong thực tế, nhưng có trường hợp xấu $\Theta(n^2)$ nếu chọn pivot không tốt.
- Quick Sort ngẫu nhiên hóa giúp tránh đầu vào cố tình gây xấu.
- Một số bài toán thực tế có thể cải thiện từ $O(n^2)$ xuống $O(n\log n)$ bằng cách sắp xếp và tìm kiếm.

---

## Bài tập nộp cuối lab

Sinh viên nộp một file notebook đã chạy đầy đủ, trong đó hoàn thành các yêu cầu:

1. Hoàn thiện và giải thích TODO 1.
2. Hoàn thiện và giải thích TODO 2.
3. Hoàn thiện và giải thích TODO 3.
4. Thực hiện thí nghiệm TODO 4 với ít nhất 4 kích thước dữ liệu khác nhau.
5. Trả lời câu hỏi TODO 5 về Karatsuba.
6. Viết ngắn gọn từ 5 đến 7 dòng:  
   **Khi nào nên dùng chia để trị? Khi nào không nên dùng?**

## Tài liệu tham khảo

- T. H. Cormen, C. E. Leiserson, R. L. Rivest, C. Stein, *Introduction to Algorithms*, 4th edition, MIT Press, 2022.
- J. Kleinberg, É. Tardos, *Algorithm Design*, Addison-Wesley, 2005.
- S. Dasgupta, C. Papadimitriou, U. Vazirani, *Algorithms*, McGraw-Hill, 2006.
- Bài giảng: **Phương pháp chia để trị**, môn Thuật toán ứng dụng.